# Train the instance detector: RF-DETR-Seg on window crops (GPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. **Run it once with `PILOT = True` first** (about 20 minutes at most): it trains on a few files for 2 epochs and prints the seconds per epoch, the peak GPU memory and the projected time of the full run. Nobody has run this notebook on a real GPU yet, so the pilot is also the check that the `rfdetr` calls below are right [VERIFY].

This is phase F of `docs/ROADMAP_V2.md`. The detector only groups and scores the pixels of the U-Net: `notebooks/export_detections.ipynb` predicts with the checkpoint made here, and `notebooks/sweep_detector.ipynb` decides on the holdout whether the fusion beats the current pipeline.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.detector_train`. An older version stops at the first code cell.

**Settings**

- Accelerator: **GPU T4**, a single T4. The notebook stops if there is no CUDA GPU or under 10 GiB of memory.
- Internet: **On**. It runs `pip install "rfdetr[train,loggers]==1.12.0"` and `rfdetr` downloads the pretrained weights `rf-detr-seg-large.pt` (the library's config names the license of both as Apache-2.0; the notebook prints the package's license text and stops if it is not Apache).
- Everything a run may change is in the settings cell: `PILOT`, `MODEL_CLASS`, `RESOLUTION`, `EPOCHS`, `BATCH_SIZE`, `MAX_TRAIN_HOURS`, `PILOT_STOP_AFTER_MINUTES`. `EPOCHS = 16` comes from a 2-epoch pilot at batch 4 (109 s per epoch on 40 + 10 files). `BATCH_SIZE = 2` with `GRAD_ACCUM_STEPS = 2` comes from a later 2-epoch pilot where batch 4 ran out of memory on the T4's 14.56 GiB; batch 2 peaked at 8.49 GiB reserved. The product of 4 keeps the library's default (`batch_size` 4, `grad_accum_steps` 1 in rfdetr 1.12.0). `PILOT_STOP_AFTER_MINUTES` (default `None`) makes the wall-clock guard fire after that many minutes, so a pilot can exercise the guard.

**What it does**

1. Checks the resolution against the model's patch size and window count, read from its config. `RESOLUTION = 768` is not the Large default (504), so it is passed explicitly; the library raising on it is the signal to stop.
2. `build_detector_splits` with its defaults (holdout = the U-Net's date holdout, detector validation = the newest tenth of the dates left, the rest train) and `write_detector_dataset` (1152 px windows resized to the resolution, one COCO folder per split). It logs how many annotations `min_area = 100` drops and how many windows are negative examples. The holdout is never written to disk.
3. Checks the written folders with `filament.detector_train.check_coco_folder` and `pycocotools`, then trains. The library's own loader runs at the start of training, so a layout error stops the run before the first epoch [VERIFY: the single category with id 1 is assumed to be accepted; if the pilot stops on it, fix `coco_payload` in `src` with a test].
4. Trains with early stopping and a wall-clock guard (`MAX_TRAIN_HOURS`; `PILOT_MAX_MINUTES` in a pilot; `PILOT_STOP_AFTER_MINUTES` when set). The library writes its checkpoints to the scratch disk. In the 2-epoch pilot it wrote `checkpoint_best_total.pth`, `checkpoint_best_ema.pth` and `last_ema.pth`, and no per-epoch `checkpoint.pth` (its `checkpoint_interval` is 10). A watcher copies the current best checkpoint to the saved output each time it changes (temp file plus `os.replace`), so a Kaggle kill, a CUDA OOM or the guard still leave weights in the output. The train cell survives `KeyboardInterrupt`, `SystemExit` and any exception, logs what happened and lets the checkpoint cell run; if nothing was written at all, that cell fails loudly.
5. Picks the checkpoint (`CHECKPOINT_CHOICE`: the library's **best on the detector validation folder**, or the **last EMA weights**, `last_ema.pth`), copies it to the output, and prints its SHA-256.

**Output:** `/kaggle/working/detector_<MODEL_CLASS>_<RESOLUTION>.pth` (`_pilot` is added to the name in a pilot), `detector_train_manifest.json` and a few small log files in `run_logs/`. Do not commit the weights. Save this notebook as `train-detector` so that `export_detections.ipynb` finds the checkpoint.

It does not read the holdout (only its names and dates, to refuse an overlap), does not read the test images, does not write `submission.csv` and does not submit to Kaggle.

## 1. `src` on the path

`src` is the directory that contains `filament/detector_train.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path

MARKER = Path("filament") / "detector_train.py"


def _local_src() -> Path | None:
    """A checkout that already has the module this notebook needs (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / MARKER).is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    The marker may sit at any depth. Directories named ``competitions`` are not
    walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / MARKER).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the private
    dataset's ``src`` is used in place. It is not copied to ``/kaggle/working``: a
    copy there would be saved with this notebook's output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            f"Could not find src/{MARKER.as_posix()} in the working directory or "
            "under /kaggle/input (any depth outside competitions). Attach the "
            "dataset ignacioaguirrecandia/solar-code. If it is attached, its "
            "version predates the detector modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings

Everything a run may change is in the next cell. The values are untuned: `EPOCHS = 16` comes from a projection at batch 4 (about 21 to 24 minutes per full epoch, 16 epochs about 6.3 h), and `BATCH_SIZE = 2` with `GRAD_ACCUM_STEPS = 2` from a batch 4 out-of-memory pilot. Neither comes from a full run, batch 2 speed was not measured before the full run, and `MAX_TRAIN_HOURS = 7.0` leaves two hours of the 9-hour session for the writing of the dataset and the export. `CHECKPOINT_CHOICE = "best"` takes the library's best checkpoint on the detector validation folder, never the holdout.

In [ ]:
# ---- Settings. Nothing below this cell needs editing. -----------------------
PILOT = False  # True: 40 train files, 2 epochs, stops within 20 minutes, prints a projection
MODEL_CLASS = "RFDETRSegLarge"
RESOLUTION = 768  # the Large default is 504; a multiple of patch_size * num_windows
EPOCHS = 16
BATCH_SIZE = 2  # 4 ran out of memory on a T4 (14.56 GiB) at resolution 768
GRAD_ACCUM_STEPS = 2  # effective batch 4, the library default
MAX_TRAIN_HOURS = 7.0  # wall-clock guard on the train call
CHECKPOINT_CHOICE = (
    "best"  # "best": library's best on detector val; "last": last_ema.pth
)
EARLY_STOPPING_PATIENCE = 5  # epochs without a better validation mAP; 0 switches it off
SEED = 0
RFDETR_VERSION = "1.12.0"

# ---- Used only when PILOT is True -------------------------------------------
PILOT_TRAIN_FILES = 40
PILOT_VAL_FILES = 10
PILOT_EPOCHS = 2
PILOT_MAX_MINUTES = 20.0
PILOT_STOP_AFTER_MINUTES = None  # a number makes the guard fire after that many minutes

MIN_GPU_GIB = 10.0

## 3. GPU, `rfdetr` and its license

The install comes before the first `import torch` so that a changed `torch` is the one that gets imported. The license text of the package is printed in full and the run stops unless it is Apache. The model config names the license of the pretrained weights too.

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"rfdetr[train,loggers]=={RFDETR_VERSION}",
    ],
    check=True,
)

In [ ]:
import _thread
import dataclasses
import json
import shutil
import tempfile
import threading
import time
import traceback
from importlib import metadata

import rfdetr
import rfdetr.config as rfdetr_config
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Set the accelerator to GPU T4 and run again.")
GPU = torch.cuda.get_device_properties(0)
GPU_GIB = GPU.total_memory / 2**30
print(f"gpu={GPU.name} memory={GPU_GIB:.1f} GiB torch={torch.__version__}")
if GPU_GIB < MIN_GPU_GIB:
    raise RuntimeError(f"GPU memory {GPU_GIB:.1f} GiB is under {MIN_GPU_GIB} GiB.")

distribution = metadata.distribution("rfdetr")
if distribution.version != RFDETR_VERSION:
    raise RuntimeError(
        f"rfdetr {distribution.version} is installed, not {RFDETR_VERSION}"
    )
print(
    "rfdetr",
    distribution.version,
    "| License:",
    distribution.metadata.get("License"),
    "| License-Expression:",
    distribution.metadata.get("License-Expression"),
)
license_texts = []
for entry in distribution.files or []:
    if "LICEN" in entry.name.upper():
        path = Path(distribution.locate_file(entry))
        if path.is_file():
            license_texts.append(path.read_text(encoding="utf-8", errors="replace"))
if not license_texts:
    raise RuntimeError("The rfdetr distribution has no license file to print.")
for text in license_texts:
    print(text)
if not any("Apache License" in text for text in license_texts):
    raise RuntimeError("The rfdetr license is not Apache. Stop and ask Nacho.")

model_class = getattr(rfdetr, MODEL_CLASS)
config_class = getattr(rfdetr_config, f"{MODEL_CLASS}Config")
fields = config_class.model_fields
PATCH_SIZE = fields["patch_size"].default
NUM_WINDOWS = fields["num_windows"].default
print(
    f"{MODEL_CLASS}: default resolution={fields['resolution'].default} "
    f"patch_size={PATCH_SIZE} num_windows={NUM_WINDOWS} "
    f"weights license={fields['license'].default}"
)
if fields["license"].default != "Apache-2.0":
    raise RuntimeError("The model's weights are not Apache-2.0. Stop and ask Nacho.")

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.crop_data import read_gray
from filament.detector_data import (
    DEFAULT_MIN_LABEL_AREA,
    MIN_VISIBLE,
    build_detector_splits,
    frame_windows,
    split_dates,
    write_detector_dataset,
)
from filament.detector_train import (
    CHECKPOINT_CHOICES,
    check_coco_folder,
    check_resolution,
    copy_atomic,
    dataset_report,
    guard_seconds,
    pick_checkpoint,
    project_full_run,
    sha256_file,
    subsample_evenly,
    sync_best_checkpoint,
)

if CHECKPOINT_CHOICE not in CHECKPOINT_CHOICES:
    raise ValueError(f"CHECKPOINT_CHOICE must be one of {CHECKPOINT_CHOICES}")
# Fails loudly, before any data is written, if the size cannot be used.
check_resolution(RESOLUTION, PATCH_SIZE, NUM_WINDOWS)
if RESOLUTION != fields["resolution"].default:
    print(
        f"note: RESOLUTION {RESOLUTION} is not the {MODEL_CLASS} default "
        f"{fields['resolution'].default}; it is passed to the model explicitly."
    )

OUTPUT_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "detector_output"
)
SCRATCH = (
    Path("/kaggle/temp")
    if Path("/kaggle/temp").is_dir()
    else Path(tempfile.gettempdir())
)
DATA_ROOT = SCRATCH / "detector_data"
RUN_DIR = SCRATCH / "detector_run"
SUFFIX = "_pilot" if PILOT else ""
CHECKPOINT_PATH = OUTPUT_DIR / f"detector_{MODEL_CLASS}_{RESOLUTION}{SUFFIX}.pth"
MANIFEST_PATH = OUTPUT_DIR / "detector_train_manifest.json"
# The watcher's safety copy of the best checkpoint so far. With "last" it gets its own
# name, so a killed run never leaves a best checkpoint under the name of a last one.
WATCH_PATH = (
    CHECKPOINT_PATH
    if CHECKPOINT_CHOICE == "best"
    else CHECKPOINT_PATH.with_name(CHECKPOINT_PATH.stem + "_best_so_far.pth")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DIR.exists() and any(RUN_DIR.glob("*.pth")):
    raise RuntimeError(
        f"{RUN_DIR} already holds checkpoints from an earlier run. Use a fresh session."
    )
print(f"output={OUTPUT_DIR} scratch={SCRATCH}")

## 4. Splits, labels and the dataset on disk

`build_detector_splits` is called with its defaults and nothing else. In a pilot the train and validation lists are thinned evenly afterwards (`subsample_evenly`), so a pilot sees every era of the data. The holdout is not touched: `write_detector_dataset` only reads its names and dates, to refuse an overlap, and raises if a holdout file or date is in train or validation.

The report counts, at the labels' native resolution and before the resize, how many annotations `min_area = 100` drops and how many windows are negative examples (a view with no filament in a window).

In [ ]:
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
full_splits = build_detector_splits(loaded)
print(
    f"files: train={len(full_splits.train)} val={len(full_splits.val)} "
    f"holdout={len(full_splits.holdout)} (the holdout is not written or read)"
)
for name, images in (("train", full_splits.train), ("val", full_splits.val)):
    days = sorted(split_dates(images))
    print(
        f"  {name}: {len(images)} files, dates {days[0]} .. {days[-1]} ({len(days)} days)"
    )

splits = full_splits
if PILOT:
    splits = full_splits._replace(
        train=subsample_evenly(full_splits.train, PILOT_TRAIN_FILES),
        val=subsample_evenly(full_splits.val, PILOT_VAL_FILES),
    )
    print(f"PILOT: train={len(splits.train)} files, val={len(splits.val)} files")

WINDOWS = frame_windows()
print(
    f"windows per frame: {len(WINDOWS)} of {WINDOWS[0].width} px, resized to {RESOLUTION}"
)
REPORTS = {}
for name, images in (("train", splits.train), ("val", splits.val)):
    report = dataset_report(images, WINDOWS, target=RESOLUTION)
    REPORTS[name] = report
    print(
        f"  {name}: {report.examples} examples (file x window x annotator view), "
        f"{report.negative} negative ({report.negative_fraction:.1%}), "
        f"{report.instances} instances kept, "
        f"{report.dropped_by_min_area} dropped by min_area={DEFAULT_MIN_LABEL_AREA} "
        f"(min_visible={MIN_VISIBLE})"
    )


def load_frame(image):
    return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))


started = time.monotonic()
counts = write_detector_dataset(
    DATA_ROOT, splits, load_frame, WINDOWS, target=RESOLUTION
)
for folder, count in counts.items():
    print(f"  wrote {folder}: {count.files} images, {count.instances} instances")
print(f"dataset written in {time.monotonic() - started:.0f} s")

## 5. Check the folders

`check_coco_folder` checks files, ids, polygons and boxes without the library. Then `pycocotools`, which the library's evaluation uses, loads each folder and turns 25 annotations into masks. A single category with id 1 is what `coco_payload` writes; the library's own dataset code runs when training starts, and a complaint about the category is the first thing to look for if the pilot stops there.

In [ ]:
import cv2
from pycocotools.coco import COCO

for folder in ("train", "valid"):
    path = DATA_ROOT / folder
    check = check_coco_folder(path)
    print(
        f"{folder}: {check.images} images, {check.annotations} annotations, "
        f"{check.negative_images} negative, categories {check.category_ids}"
    )
    coco = COCO(str(path / "_annotations.coco.json"))
    for annotation in coco.loadAnns(coco.getAnnIds()[:25]):
        image = coco.imgs[annotation["image_id"]]
        mask = coco.annToMask(annotation)
        if mask.shape != (RESOLUTION, RESOLUTION) or not mask.any():
            raise RuntimeError(
                f"{folder}: annotation {annotation['id']} gives a bad mask {mask.shape}"
            )
    first = cv2.imread(str(path / coco.imgs[min(coco.imgs)]["file_name"]))
    if first is None or first.shape != (RESOLUTION, RESOLUTION, 3):
        raise RuntimeError(
            f"{folder}: first image has shape {None if first is None else first.shape}"
        )

## 6. Train

The model is built with `resolution=RESOLUTION`; the library rejecting it stops the run. The train arguments are checked against the library's `SegmentationTrainConfig` fields first, so a renamed option fails here and not an hour in.

- **Wall-clock guard.** A timer interrupts the main thread when `MAX_TRAIN_HOURS` (or `PILOT_MAX_MINUTES` in a pilot, or `PILOT_STOP_AFTER_MINUTES` when set) has passed. The library is expected to stop cleanly and keep the checkpoints it has written; the notebook says so when the guard fired. Whatever stops the `train` call (the guard as `KeyboardInterrupt` or `SystemExit`, a manual interrupt, a CUDA OOM or any other exception) is caught and printed with its traceback, and the next cell still runs. It raises if no checkpoint was written at all. To exercise the guard, run a pilot with `PILOT_STOP_AFTER_MINUTES` of about 3, after the first validation has written a checkpoint.
- **Checkpoints.** `output_dir` of the library is on the scratch disk. A watcher thread polls it every minute, prints each time a `.pth` file there changes, and copies the current best checkpoint to the saved output when it changes (temp file plus `os.replace`, so the copy is always complete).
- **Early stopping** follows `EARLY_STOPPING_PATIENCE` on the library's validation mAP, which is computed on the `valid` folder, the detector validation dates.

In [ ]:
model = model_class(resolution=RESOLUTION)
built = getattr(getattr(model, "model_config", None), "resolution", None)
if built is None:
    print(
        "WARNING: could not read the resolution back from the model; relying on the library's check [VERIFY]."
    )
elif built != RESOLUTION:
    raise RuntimeError(f"The model was built at {built}, not {RESOLUTION}.")
else:
    print(f"model built at resolution {built}")

epochs = PILOT_EPOCHS if PILOT else EPOCHS
train_kwargs = {
    "dataset_dir": str(DATA_ROOT),
    "output_dir": str(RUN_DIR),
    "epochs": epochs,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "early_stopping": EARLY_STOPPING_PATIENCE > 0,
    "early_stopping_patience": max(EARLY_STOPPING_PATIENCE, 1),
    "seed": SEED,
    "num_workers": 2,
    "tensorboard": False,
    "run_test": False,
}
unknown = sorted(
    set(train_kwargs) - set(rfdetr_config.SegmentationTrainConfig.model_fields)
)
if unknown:
    raise RuntimeError(f"rfdetr {RFDETR_VERSION} has no train option(s) {unknown}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

stop_watching = threading.Event()


def _watch() -> None:
    seen: dict[str, float] = {}
    copied: dict[str, tuple[int, int]] = {}
    while not stop_watching.wait(60.0):
        for path in sorted(RUN_DIR.glob("*.pth")):
            stat = path.stat()
            if seen.get(path.name) != stat.st_mtime:
                seen[path.name] = stat.st_mtime
                print(
                    f"[checkpoint] {path.name} {stat.st_size / 2**20:.0f} MiB "
                    f"at {time.strftime('%H:%M:%S')}",
                    flush=True,
                )
        try:
            name = sync_best_checkpoint(RUN_DIR, WATCH_PATH, copied)
        except Exception as error:  # noqa: BLE001 - a failed copy must not end the watcher
            print(f"[checkpoint] copy to the output failed: {error!r}", flush=True)
        else:
            if name:
                print(f"[checkpoint] {name} copied to {WATCH_PATH.name}", flush=True)


guard_fired = threading.Event()


def _guard() -> None:
    guard_fired.set()
    _thread.interrupt_main()


limit_seconds = guard_seconds(
    pilot=PILOT,
    max_train_hours=MAX_TRAIN_HOURS,
    pilot_max_minutes=PILOT_MAX_MINUTES,
    stop_after_minutes=PILOT_STOP_AFTER_MINUTES,
)
watcher = threading.Thread(target=_watch, daemon=True)
timer = threading.Timer(limit_seconds, _guard)
timer.daemon = True
torch.cuda.reset_peak_memory_stats()
print(
    f"training {epochs} epochs, guard at {limit_seconds / 3600:.2f} h, kwargs={train_kwargs}",
    flush=True,
)
started = time.monotonic()
train_error = None  # set when train() stopped for any reason other than the guard
watcher.start()
timer.start()
try:
    model.train(**train_kwargs)
except (KeyboardInterrupt, SystemExit, Exception) as error:  # noqa: BLE001
    # Whatever stops train() (the guard, a Lightning exit, an OOM at epoch 15) must not
    # skip the checkpoint cell: the checkpoints on the scratch disk are what is left.
    if guard_fired.is_set() and isinstance(error, (KeyboardInterrupt, SystemExit)):
        print(f"train() was stopped by the wall-clock guard ({type(error).__name__}).")
    else:
        train_error = f"{type(error).__name__}: {error}"
        print(f"TRAINING STOPPED EARLY by {train_error}", flush=True)
        traceback.print_exc()
finally:
    timer.cancel()
    stop_watching.set()
    watcher.join(timeout=300)
train_seconds = time.monotonic() - started
PEAK_ALLOCATED_GIB = torch.cuda.max_memory_allocated() / 2**30
PEAK_RESERVED_GIB = torch.cuda.max_memory_reserved() / 2**30
if guard_fired.is_set():
    print(
        f"STOPPED by the wall-clock guard after {train_seconds / 60:.1f} min; the checkpoints on disk are used."
    )
print(
    f"train call: {train_seconds / 60:.1f} min, peak GPU memory {PEAK_ALLOCATED_GIB:.2f} GiB allocated, {PEAK_RESERVED_GIB:.2f} GiB reserved"
)
print(
    "run dir:",
    {
        p.name: f"{p.stat().st_size / 2**20:.0f} MiB"
        for p in sorted(RUN_DIR.glob("*.pth"))
    },
)
print(f"disk free on scratch: {shutil.disk_usage(SCRATCH).free / 2**30:.1f} GiB")

if PILOT:
    files_pilot = len(splits.train) + len(splits.val)
    files_full = len(full_splits.train) + len(full_splits.val)
    # Train and validation files are scaled separately, validation at no cost: the
    # longest projection (docs: project_full_run).
    projection = project_full_run(
        train_seconds,
        PILOT_EPOCHS,
        len(splits.train),
        len(splits.val),
        len(full_splits.train),
        len(full_splits.val),
        EPOCHS,
    )
    print(
        f"PILOT seconds per epoch: {projection.seconds_per_epoch:.0f} (includes start-up and downloads, so it leans long)"
    )
    print(
        f"PILOT peak GPU memory: {PEAK_ALLOCATED_GIB:.2f} GiB allocated, {PEAK_RESERVED_GIB:.2f} GiB reserved at batch size {BATCH_SIZE}"
    )
    fits = (
        "fits in"
        if projection.projected_hours <= MAX_TRAIN_HOURS
        else "DOES NOT FIT in"
    )
    print(
        f"PILOT projected full run: {projection.projected_hours:.1f} h for {EPOCHS} epochs of "
        f"{files_full} files ({projection.projected_epoch_seconds / 60:.0f} min per epoch); "
        f"{fits} MAX_TRAIN_HOURS={MAX_TRAIN_HOURS}"
    )
    if guard_fired.is_set() or train_error:
        print("PILOT: training did not finish, so the projection above is not valid.")

## 7. The checkpoint

`CHECKPOINT_CHOICE = "best"` is the library's best checkpoint, chosen on the `valid` folder: the detector validation dates, which the holdout does not touch. `"last"` is `last_ema.pth`, the EMA weights after the last epoch that ran (the library's `checkpoint.pth` is written every 10 epochs and is never used). The notebook raises if training stopped with no checkpoint written, and otherwise fails with a listing of the directory if the library names its files differently [VERIFY on the pilot]. The chosen file is copied to the output and its SHA-256 is printed; keep it with the run's notes.

In [ ]:
names = [path.name for path in RUN_DIR.glob("*.pth")]
if not names:
    raise RuntimeError(
        f"No checkpoint was written to {RUN_DIR}; training "
        + (f"stopped with {train_error}." if train_error else "wrote nothing.")
    )
if train_error:
    print(
        f"WARNING: training stopped early with {train_error}; using the checkpoints it wrote."
    )
chosen_name = pick_checkpoint(names, CHECKPOINT_CHOICE)
chosen = RUN_DIR / chosen_name
meaning = (
    "the library's best on the detector validation folder (never the holdout)"
    if CHECKPOINT_CHOICE == "best"
    else "the last EMA weights (last_ema.pth), from the last epoch that ran"
)
print(f"checkpoint used: {chosen_name} = {meaning}")
copy_atomic(chosen, CHECKPOINT_PATH)
digest = sha256_file(CHECKPOINT_PATH)

logs = OUTPUT_DIR / "run_logs"
logs.mkdir(exist_ok=True)
for path in RUN_DIR.iterdir():
    if path.is_file() and path.suffix != ".pth" and path.stat().st_size < 2 * 2**20:
        shutil.copy2(path, logs / path.name)

manifest = {
    "pilot": PILOT,
    "model_class": MODEL_CLASS,
    "resolution": RESOLUTION,
    "patch_size": PATCH_SIZE,
    "num_windows": NUM_WINDOWS,
    "epochs_requested": epochs,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "seed": SEED,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "stopped_by_guard": guard_fired.is_set(),
    "train_error": train_error,
    "train_seconds": round(train_seconds, 1),
    "peak_gpu_gib_allocated": round(PEAK_ALLOCATED_GIB, 3),
    "checkpoint_choice": CHECKPOINT_CHOICE,
    "checkpoint_source": chosen_name,
    "checkpoint_file": CHECKPOINT_PATH.name,
    "checkpoint_sha256": digest,
    "rfdetr": RFDETR_VERSION,
    "torch": torch.__version__,
    "gpu": GPU.name,
    "files": {
        "train": len(splits.train),
        "val": len(splits.val),
        "holdout_not_used": len(full_splits.holdout),
    },
    "reports": {name: dataclasses.asdict(report) for name, report in REPORTS.items()},
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
)
print(f"wrote {MANIFEST_PATH}")
print(f"CHECKPOINT {CHECKPOINT_PATH} sha256={digest}")
print(
    "This notebook read no holdout frame or label, no test image, wrote no submission and did not submit."
)